# Function 6: 5D black-box maximisation

You’re optimising a cake recipe using a black-box function with five ingredient inputs, for example flour, sugar, eggs, butter and milk. Each recipe is evaluated with a combined score based on flavour, consistency, calories, waste and cost, where each factor contributes negative points as judged by an expert taster. This means the total score is negative by design. To frame this as a maximisation problem, your goal is to bring that score as close to zero as possible or, equivalently, to maximise the negative of the total sum.

## How this notebook works

1. **Data loading:** observations are read from `data.csv`; the incumbent is the best observed `y`.
2. **Gaussian process:** an ARD Matérn 5/2 kernel plus a white-noise term, fitted to standardised `y`. Length scales are bounded to 0.05–2.0 and noise to 0.001–1.0, so no input can go flat and the noise estimate cannot collapse to zero.
3. **Trust-region replay:** the TuRBO box length is rebuilt from the weekly history: it doubles after `SUCC_TOL` consecutive improvements and halves after `FAIL_TOL` failures.
4. **Candidates:** Sobol points in an ARD-shaped box around the incumbent (each perturbing a random subset of coordinates), plus local perturbations of the incumbent.
5. **Scoring:** a Thompson sample, the posterior-mean maximiser and the Expected Improvement maximiser, each reported with its GP prediction.
6. **Walk-forward calibration check:** from round 8 on, the GP is refitted on the earlier rounds and asked to predict each round. Standardised errors above 2 in absolute value mean the model is overconfident.
7. **Final-round selection:** two local candidates are compared under four GP specifications (Matérn 5/2 and 3/2 with fitted noise, RBF, and Matérn 5/2 with low fixed noise): **A**, the best posterior mean within ±0.01 of the incumbent, and **B**, the point with the highest probability of beating the best observed `y` within ±0.04. The candidate preferred by more models is recommended (ties go to the higher mean probability) and printed in submission format.

Everything is derived from `data.csv`: append new rows and re-run, with no code edits.

**Data-handling choices.**

- *Noise and calibration.* Function 6 is noisy, so the white-noise level is fitted (within bounds) and EI and predictive standard deviations include it. The walk-forward check shows the GP's stated uncertainty has been too small in places, which is why the final round compares several GP specifications instead of trusting one. Agreement between posterior mean and EI is consistency within one model, not independent confirmation.
- *Trust region.* The TuRBO state is reconstructed from the history, not maintained by a fully sequential run. Steps 2 to 5 still print the original TuRBO candidates; the final recommendation comes from step 7.
- *Input format.* Candidates are truncated (not rounded) to 6 decimals, as the portal requires.

In [1]:
# =============================================================================
# IMPORT LIBRARIES
# =============================================================================
# Core numerics, scikit-learn Gaussian process regression and Sobol sampling.

import warnings
from decimal import Decimal, ROUND_DOWN

import numpy as np
import pandas as pd
from scipy.stats import qmc, norm
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, RBF, WhiteKernel

warnings.filterwarnings("ignore", module="sklearn")

In [2]:
# =============================================================================
# DATA LOADING
# =============================================================================
# Loads the centrally prepared BBO dataset and selects the active function.
# Active input dimensions are inferred from the x-columns that contain data.

RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)

DATA_PATH = "../data/data.csv"
FUNCTION_ID = 6

# round_trip parsing keeps float64 values exactly as written in the CSV
data = pd.read_csv(DATA_PATH, float_precision="round_trip")
function_data = data[data["function"] == FUNCTION_ID].copy()

feature_cols = [
    f"x{i}" for i in range(1, 9)
    if function_data[f"x{i}"].notna().any()
]

# Modelling arrays
X = function_data[feature_cols].to_numpy()
y = function_data["y"].to_numpy()

# Display dataframe
df = function_data[[*feature_cols, "y", "round"]].copy()
df["Stage"] = df["round"].apply(
    lambda r: "Initial Data" if r == 0 else f"Week {r}"
)
df = df.rename(columns={"round": "Round"})
df = df[[*feature_cols, "y", "Stage", "Round"]]

# Display formatting only — underlying float64 values are unchanged
pd.set_option(
    "display.float_format",
    lambda v: f"{v:.6e}" if abs(v) < 1e-3 and v != 0 else f"{v:.6f}"
)

print(f"Function {FUNCTION_ID} observations: {len(df)}")
print(f"Active features: {feature_cols}")
print(f"Dimensions: {len(feature_cols)}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Rounds: R0–R{df['Round'].max()}")

display(df)

Function 6 observations: 32
Active features: ['x1', 'x2', 'x3', 'x4', 'x5']
Dimensions: 5
X shape: (32, 5)
y shape: (32,)
Rounds: R0–R12


,x1,x2,x3,x4,x5,y,Stage,Round
85,0.728186,0.154693,0.732552,0.693997,0.056401,-0.714265,Initial Data,0
86,0.242384,0.844100,0.577809,0.679021,0.501953,-1.209955,Initial Data,0
87,0.729523,0.748106,0.679775,0.356552,0.671054,-1.672200,Initial Data,0
88,0.770620,0.114404,0.046780,0.648324,0.273549,-1.536058,Initial Data,0
89,0.618812,0.331802,0.187288,0.756238,0.328835,-0.829237,Initial Data,0
90,0.784958,0.910682,0.708120,0.959225,0.004911,-1.247049,Initial Data,0
91,0.145111,0.896685,0.896322,0.726272,0.236272,-1.233786,Initial Data,0
92,0.945069,0.288459,0.978806,0.961656,0.598016,-1.694343,Initial Data,0
93,0.125720,0.862725,0.028544,0.246605,0.751206,-2.571170,Initial Data,0
94,0.757594,0.355831,0.016523,0.434207,0.112433,-1.309116,Initial Data,0


In [3]:
# =============================================================================
# TuRBO-1: BAYESIAN OPTIMISATION
# =============================================================================
# Proposes the next Function 6 query (maximisation on [0, 1]^d).
#   1. Fit an ARD Matern-5/2 Gaussian process to all observations, with
#      bounded lengthscales and a noise floor.
#   2. Replay the weekly history to rebuild the trust-region box length
#      (double after SUCC_TOL successes, halve after FAIL_TOL failures).
#   3. Centre an ARD-shaped box on the best observed point, then generate
#      Sobol candidates inside it plus local perturbations of the incumbent.
#   4. Score candidates with the GP: Thompson sample, posterior mean and
#      Expected Improvement (EI). The EI maximiser is the recommended query.

# -----------------------------------------------------------------------------
# SETTINGS
# -----------------------------------------------------------------------------

LENGTH_INIT = 0.08     # initial box side length (fraction of the unit cube)
LENGTH_MIN = 0.5 ** 7  # box is reset to LENGTH_INIT if it shrinks below this
LENGTH_MAX = 0.40
SUCC_TOL = 3           # consecutive improving rounds before the box doubles
FAIL_TOL = None        # None -> max(4, d) failing rounds before the box halves
SUCCESS_EPS = 1e-6     # minimum gain that counts as an improvement

N_CAND = 8192          # Sobol candidates inside the trust region
N_LOCAL = 2048         # Gaussian perturbations around the incumbent
LOCAL_SCALE = 0.015    # std of those perturbations
N_TS = 1024            # candidates entering the Thompson-sampling draw

LS_BOUNDS = (0.05, 2.0)      # lengthscale bounds: stops any dimension going flat
NOISE_BOUNDS = (1e-3, 1.0)   # noise bounds on standardised y: stops a near-zero noise fit


# -----------------------------------------------------------------------------
# HELPERS
# -----------------------------------------------------------------------------

def truncate6(value):
    """Truncate (not round) to six decimals, as required by the submission format."""
    return str(
        Decimal(repr(float(value))).quantize(Decimal("0.000001"), rounding=ROUND_DOWN)
    )


def format_point(x):
    """Format a point as the dash-separated submission string."""
    return "-".join(truncate6(v) for v in x)


def make_gp(d, seed):
    """Build the GP used everywhere (main run, calibration, final pick)."""
    kernel = (
        ConstantKernel(1.0, (1e-2, 1e2))
        * Matern(length_scale=[0.3] * d, length_scale_bounds=LS_BOUNDS, nu=2.5)
        + WhiteKernel(noise_level=0.1, noise_level_bounds=NOISE_BOUNDS)
    )
    return GaussianProcessRegressor(
        kernel=kernel, normalize_y=False, n_restarts_optimizer=15, random_state=seed
    )


# -----------------------------------------------------------------------------
# TuRBO-1
# -----------------------------------------------------------------------------

def turbo_next(X, y, rounds, x_cols, seed=42):
    """Return the next-query candidates for observations (X, y) taken in `rounds`."""
    n, d = X.shape
    rng = np.random.default_rng(seed)
    fail_tol = max(4, d) if FAIL_TOL is None else FAIL_TOL

    # 1. Fit the Gaussian process on standardised y
    y_mu, y_sd = y.mean(), y.std()
    y_scaled = (y - y_mu) / y_sd

    gp = make_gp(d, seed).fit(X, y_scaled)

    ls = np.atleast_1d(gp.kernel_.k1.k2.length_scale)
    noise_sd = np.sqrt(gp.kernel_.k2.noise_level) * y_sd

    # 2. Replay the weekly history to reconstruct the trust-region length
    order = np.argsort(rounds, kind="stable")
    r_sorted, y_sorted = rounds[order], y[order]

    first_round = r_sorted.min()
    best = y_sorted[r_sorted == first_round].max()
    length, succ, fail = LENGTH_INIT, 0, 0

    print("Round | Best y       | Result  | Box length")
    for r in np.unique(r_sorted[r_sorted > first_round]):
        yr = y_sorted[r_sorted == r].max()
        improved = yr > best + SUCCESS_EPS

        if improved:
            best = yr
            succ, fail = succ + 1, 0
        else:
            succ, fail = 0, fail + 1

        if succ >= SUCC_TOL:
            length = min(2 * length, LENGTH_MAX)
            succ = 0
        elif fail >= fail_tol:
            length /= 2
            fail = 0

        if length < LENGTH_MIN:
            length, succ, fail = LENGTH_INIT, 0, 0

        print(f"{r:>5} | {yr:>12.9f} | {'Success' if improved else 'Failure':<7} | {length:.4f}")

    # 3. ARD-weighted trust region centred on the best observed point
    incumbent = X[np.argmax(y)].copy()
    weights = ls / ls.mean()
    weights /= np.prod(weights) ** (1 / d)
    lb = np.clip(incumbent - weights * length / 2, 0, 1)
    ub = np.clip(incumbent + weights * length / 2, 0, 1)

    # 4. Candidates: Sobol points (each perturbing a random subset of
    #    coordinates), local perturbations of the incumbent, and the incumbent
    sobol = qmc.Sobol(d=d, scramble=True, seed=seed)
    perturbed = lb + (ub - lb) * sobol.random(N_CAND)

    mask = rng.random((N_CAND, d)) <= min(1.0, 20.0 / d)
    empty = np.where(~mask.any(axis=1))[0]
    if len(empty):
        mask[empty, rng.integers(d, size=len(empty))] = True

    sobol_candidates = np.where(mask, perturbed, incumbent)
    local = np.clip(incumbent + rng.normal(0, LOCAL_SCALE, size=(N_LOCAL, d)), lb, ub)
    candidates = np.vstack([sobol_candidates, local, incumbent[None, :]])

    # 5. GP predictions and Expected Improvement (original y scale)
    mu_scaled, sd_scaled = gp.predict(candidates, return_std=True)
    mu, sd = mu_scaled * y_sd + y_mu, sd_scaled * y_sd

    improvement = mu - y.max()
    z = np.divide(improvement, sd, out=np.zeros_like(sd), where=sd > 0)
    ei = improvement * norm.cdf(z) + sd * norm.pdf(z)
    ei = np.where(sd > 0, ei, np.maximum(improvement, 0))

    # 6. Thompson sampling over the EI/mean maximisers plus a random subset
    mean_idx, ei_idx = np.argmax(mu), np.argmax(ei)
    priority = np.unique([ei_idx, mean_idx])
    remaining = np.setdiff1d(np.arange(len(candidates)), priority)
    n_extra = min(N_TS - len(priority), len(remaining))
    selected = np.r_[priority, rng.choice(remaining, n_extra, replace=False)]

    ts_draw = gp.sample_y(candidates[selected], n_samples=1, random_state=seed).ravel()
    ts_idx = selected[np.argmax(ts_draw)]

    # 7. Report each candidate (truncated to six decimals) with its GP prediction
    def report(label, idx):
        x = np.array([float(truncate6(v)) for v in candidates[idx]])
        pred_scaled, std_scaled = gp.predict(x[None, :], return_std=True)
        print(f"\n{label}")
        print(format_point(x))
        print(f"Predicted y: {pred_scaled[0] * y_sd + y_mu:.9f} +/- {std_scaled[0] * y_sd:.9f}")
        print(f"Distance from incumbent: {np.linalg.norm(x - incumbent):.6f}")
        return x

    print(f"\nObservations: {n} | Best observed y: {y.max():.9f}")
    print(f"Incumbent: {format_point(incumbent)}")
    print(f"Kernel: {gp.kernel_}")
    print(f"GP fitted noise SD: {noise_sd:.6f}")
    print(f"Box length: {length:.4f} (success {succ}/{SUCC_TOL}, failure {fail}/{fail_tol})")
    print("\nTrust region:")
    for name, lo, hi in zip(x_cols, lb, ub):
        print(f"  {name}: [{lo:.4f}, {hi:.4f}]")

    x_ts = report("Thompson sampling", ts_idx)
    x_mean = report("Best posterior mean", mean_idx)
    x_ei = report("Best expected improvement", ei_idx)

    print(f"\nRecommended next query (Expected Improvement):\n{format_point(x_ei)}")

    return {
        "x_next": x_ei,
        "x_next_str": [truncate6(v) for v in x_ei],
        "x_ts": x_ts,
        "x_mean": x_mean,
        "x_ei": x_ei,
        "gp": gp,
        "incumbent": incumbent,
        "length": length,
        "lb": lb,
        "ub": ub,
    }


# -----------------------------------------------------------------------------
# RUN
# -----------------------------------------------------------------------------

rounds = function_data["round"].to_numpy()

result4 = turbo_next(X, y, rounds, feature_cols, seed=RANDOM_STATE)

x_next4 = result4["x_next"]
x_next4_str = result4["x_next_str"]

Round | Best y       | Result  | Box length
    1 | -0.305729456 | Success | 0.0800
    2 | -1.175428372 | Failure | 0.0800
    3 | -0.410163523 | Failure | 0.0800
    4 | -0.757140109 | Failure | 0.0800
    5 | -0.278162833 | Success | 0.0800
    6 | -0.304925466 | Failure | 0.0800
    7 | -0.237329909 | Success | 0.0800
    8 | -0.385476478 | Failure | 0.0800
    9 | -0.068174249 | Success | 0.0800
   10 | -0.071432425 | Failure | 0.0800
   11 |  0.005978097 | Success | 0.0800
   12 | -0.347571868 | Failure | 0.0800

Observations: 32 | Best observed y: 0.005978097
Incumbent: 0.429654-0.391293-0.666659-0.751633-0.055887
Kernel: 1.38**2 * Matern(length_scale=[0.768, 0.949, 1.49, 1.14, 1.62], nu=2.5) + WhiteKernel(noise_level=0.0257)
GP fitted noise SD: 0.109248
Box length: 0.0800 (success 0/3, failure 1/5)

Trust region:
  x1: [0.4029, 0.4564]
  x2: [0.3582, 0.4243]
  x3: [0.6148, 0.7185]
  x4: [0.7120, 0.7913]
  x5: [0.0000, 0.1122]

Thompson sampling
0.406371-0.396035-0.646808-0.7848

In [4]:
# =============================================================================
# WALK-FORWARD CALIBRATION CHECK
# =============================================================================
# For each round r from 8 on, fit the GP on rounds before r and predict round r.
# z = (actual - predicted) / predicted SD. Most |z| should be below 2.
# Several |z| above 2 means the model is overconfident.

for r in range(8, int(rounds.max()) + 1):
    tr, te = rounds < r, rounds == r
    mu_, sd_ = y[tr].mean(), y[tr].std()
    g = make_gp(X.shape[1], RANDOM_STATE).fit(X[tr], (y[tr] - mu_) / sd_)
    m, s = g.predict(X[te], return_std=True)
    z = (y[te] - (m * sd_ + mu_)) / (s * sd_)
    print(f"R{r}: actual {y[te][0]:+.3f} | pred {m[0]*sd_+mu_:+.3f} +/- {s[0]*sd_:.3f} | z {z[0]:+.2f}")

R8: actual -0.385 | pred -0.292 +/- 0.049 | z -1.92
R9: actual -0.068 | pred -0.337 +/- 0.067 | z +4.02
R10: actual -0.071 | pred -0.129 +/- 0.090 | z +0.64
R11: actual +0.006 | pred -0.046 +/- 0.073 | z +0.71
R12: actual -0.348 | pred +0.048 +/- 0.078 | z -5.10


In [5]:
# =============================================================================
# FINAL-WEEK PICK: MAXIMUM PROBABILITY OF BEATING THE BEST OBSERVED y
# =============================================================================
# No later round exists to learn from, so only the best y counts.
# Candidates: Sobol points in a 0.08-wide box around the incumbent.
# Pick the one with the highest probability of exceeding the current best.

d = X.shape[1]
best = y.max()
y_mu, y_sd = y.mean(), y.std()

gp = make_gp(d, RANDOM_STATE).fit(X, (y - y_mu) / y_sd)
inc = X[np.argmax(y)]

cand = np.clip(inc + (qmc.Sobol(d, seed=0).random(8192) - 0.5) * 0.08, 0, 1)
cand = np.vstack([cand, inc])

m, s = gp.predict(cand, return_std=True)
m, s = m * y_sd + y_mu, s * y_sd
pi = 1 - norm.cdf((best - m) / s)

i = pi.argmax()
x_final = np.array([float(truncate6(v)) for v in cand[i]])

print(format_point(x_final))
print(f"mean {m[i]:+.3f} sd {s[i]:.3f} P(beat) {pi[i]:.2f}")

0.445356-0.389722-0.706185-0.751823-0.017030
mean -0.148 sd 0.123 P(beat) 0.11


In [6]:
# =============================================================================
# FINAL ROUND — GP ROBUSTNESS CHECK
# =============================================================================

d = X.shape[1]
inc = X[np.argmax(y)]
best = y.max()

# Candidate A: Conservative local perturbation
# Search within +/- 0.01 of the incumbent.

rng = np.random.default_rng(RANDOM_STATE)

local = np.clip(
    inc + rng.uniform(-0.01, 0.01, size=(4096, d)),
    0, 1
)

# Include incumbent for comparison, but exclude it from final selection
local = np.vstack([local, inc])

# Candidate B: Probability of Improvement
# Search within +/- 0.04 of the incumbent.

sobol = qmc.Sobol(d, scramble=True, seed=RANDOM_STATE)

wide = np.clip(
    inc + (sobol.random_base2(m=13) - 0.5) * 0.08,
    0, 1
)

# Fit reference GP
y_mu, y_sd = y.mean(), y.std()

gp = make_gp(d, RANDOM_STATE).fit(
    X, (y - y_mu) / y_sd
)

# Candidate A: Best posterior mean in conservative region
mu_A = gp.predict(local[:-1]) * y_sd + y_mu
A = local[np.argmax(mu_A)]

# Candidate B: Maximum probability of improvement
mu_B, sd_B = gp.predict(wide, return_std=True)

mu_B = mu_B * y_sd + y_mu
sd_B = sd_B * y_sd

pi_B = norm.sf(
    (best - mu_B) / np.maximum(sd_B, 1e-12)
)

B = wide[np.argmax(pi_B)]

candidates = np.vstack([A, B])

print("Candidate A:", format_point(A))
print("Candidate B:", format_point(B))

candidates = np.vstack([A, B])

d = X.shape[1]
best = y.max()
y_mu, y_sd = y.mean(), y.std()
y_scaled = (y - y_mu) / y_sd

# Four alternative GP specifications
models = {
    "Matern 5/2, fitted noise": (
        ConstantKernel(1.0, (1e-2, 1e2))
        * Matern([0.3] * d, (0.01, 10), nu=2.5)
        + WhiteKernel(0.1, (1e-6, 1.0))
    ),
    "Matern 3/2, fitted noise": (
        ConstantKernel(1.0, (1e-2, 1e2))
        * Matern([0.3] * d, (0.01, 10), nu=1.5)
        + WhiteKernel(0.1, (1e-6, 1.0))
    ),
    "RBF, fitted noise": (
        ConstantKernel(1.0, (1e-2, 1e2))
        * RBF([0.3] * d, (0.01, 10))
        + WhiteKernel(0.1, (1e-6, 1.0))
    ),
    "Matern 5/2, low noise": (
        ConstantKernel(1.0, (1e-2, 1e2))
        * Matern([0.3] * d, (0.01, 10), nu=2.5)
        + WhiteKernel(1e-5, "fixed")
    ),
}

results = []

for name, kernel in models.items():

    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=False,
        n_restarts_optimizer=15,
        random_state=RANDOM_STATE
    )

    gp.fit(X, y_scaled)

    mu, sd = gp.predict(candidates, return_std=True)

    mu = mu * y_sd + y_mu
    sd = sd * y_sd

    pi = norm.sf((best - mu) / np.maximum(sd, 1e-12))

    results.append({
        "Model": name,
        "Pred A": mu[0],
        "Pred B": mu[1],
        "SD A": sd[0],
        "SD B": sd[1],
        "PI A": pi[0],
        "PI B": pi[1],
        "Preferred (PI)": "A" if pi[0] > pi[1] else "B"
    })

df = pd.DataFrame(results)

print("\nGP ROBUSTNESS COMPARISON")
print(df.round(4).to_string(index=False))

print("\nMODEL PREFERENCES")
print(df["Preferred (PI)"].value_counts().to_string())

print("\nCURRENT BEST:", round(best, 9))

Candidate A: 0.435845-0.385066-0.675783-0.748318-0.047503
Candidate B: 0.445356-0.389722-0.706185-0.751823-0.017030

GP ROBUSTNESS COMPARISON
                   Model    Pred A    Pred B     SD A     SD B     PI A         PI B Preferred (PI)
Matern 5/2, fitted noise  0.005900 -0.146000 0.016100 0.037700 0.498800     0.000000              A
Matern 3/2, fitted noise  0.002200 -0.154600 0.020700 0.046200 0.428400 3.000000e-04              A
       RBF, fitted noise -0.115400 -0.142200 0.091100 0.096500 0.091300     0.062400              A
   Matern 5/2, low noise  0.005700 -0.146100 0.016400 0.037800 0.493900     0.000000              A

MODEL PREFERENCES
Preferred (PI)
A    4

CURRENT BEST: 0.005978097


In [7]:
# =============================================================================
# FINAL RECOMMENDATION — SELECT CANDIDATE WITH STRONGEST MODEL SUPPORT
# =============================================================================

# Count how many models favour each candidate
votes_A = (df["Preferred (PI)"] == "A").sum()
votes_B = (df["Preferred (PI)"] == "B").sum()

# Select candidate with majority support
# Tie-breaker: average predicted probability of improvement
if votes_A > votes_B:
    x_final = A
    selected = "A"
elif votes_B > votes_A:
    x_final = B
    selected = "B"
else:
    selected = "A" if df["PI A"].mean() >= df["PI B"].mean() else "B"
    x_final = A if selected == "A" else B

print("\n" + "=" * 60)
print(f"FINAL RECOMMENDATION: CANDIDATE {selected}")
print(f"MODEL SUPPORT: A = {votes_A}/4 | B = {votes_B}/4")
print(f"RECOMMENDED NEXT QUERY: {format_point(x_final)}")
print("=" * 60)


FINAL RECOMMENDATION: CANDIDATE A
MODEL SUPPORT: A = 4/4 | B = 0/4
RECOMMENDED NEXT QUERY: 0.435845-0.385066-0.675783-0.748318-0.047503
